# Preparación del Dataset para Fine-tuning

Lee `checkpoint.jsonl`, balancea las clases y genera los splits train/val/test.

**Estrategia de muestreo:**
- POS: 2.400 ejemplos (de ~2.763 disponibles)
- NEG: 2.400 ejemplos (de ~2.416 disponibles)
- NEU: 2.400 ejemplos (de ~5.390 disponibles)
- MIXTO: todos (~1.036 disponibles)

**Split:** 80% train / 10% val / 10% test  
**Output:** `finetune_beto/dataset/{train,val,test}.csv`

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

CHECKPOINT_PATH = Path('dataset_etiquetado/checkpoint.jsonl')
OUTPUT_DIR      = Path('dataset')
OUTPUT_DIR.mkdir(exist_ok=True)

N_POR_CLASE  = 2400   # máximo por clase (POS, NEG, NEU)
SEED         = 42
VAL_RATIO    = 0.10
TEST_RATIO   = 0.10

LABEL_MAP = {'POS': 0, 'NEG': 1, 'NEU': 2, 'MIXTO': 3}

print('Config lista.')
print(f'Checkpoint: {CHECKPOINT_PATH}')
print(f'N por clase: {N_POR_CLASE} (MIXTO: todos)')


Config lista.
Checkpoint: ..\dataset_etiquetado\checkpoint.jsonl
N por clase: 2400 (MIXTO: todos)


In [2]:
# Leer checkpoint.jsonl
registros = []
with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            label = obj.get('label', '').strip().upper()
            texto = obj.get('oracion', obj.get('texto', obj.get('sentence', ''))).strip()
            if label in LABEL_MAP and len(texto) > 10:
                registros.append({'texto': texto, 'label': label})
        except:
            pass

df = pd.DataFrame(registros)
print(f'Total registros válidos: {len(df):,}')
print()
print('Distribución original:')
for label, n in df['label'].value_counts().items():
    pct = n / len(df) * 100
    print(f'  {label:6s}: {n:5,}  ({pct:.1f}%)')


Total registros válidos: 11,605

Distribución original:
  NEU   : 5,390  (46.4%)
  POS   : 2,763  (23.8%)
  NEG   : 2,416  (20.8%)
  MIXTO : 1,036  (8.9%)


In [3]:
# Samplear por clase
np.random.seed(SEED)
partes = []

for label in ['POS', 'NEG', 'NEU', 'MIXTO']:
    subset = df[df['label'] == label]
    if label == 'MIXTO':
        muestra = subset          # todos
    else:
        n = min(N_POR_CLASE, len(subset))
        muestra = subset.sample(n=n, random_state=SEED)
    partes.append(muestra)
    print(f'  {label:6s}: {len(muestra):,} ejemplos seleccionados')

df_bal = pd.concat(partes).sample(frac=1, random_state=SEED).reset_index(drop=True)
df_bal['label_id'] = df_bal['label'].map(LABEL_MAP)

print(f'\nTotal dataset balanceado: {len(df_bal):,}')


  POS   : 2,400 ejemplos seleccionados
  NEG   : 2,400 ejemplos seleccionados
  NEU   : 2,400 ejemplos seleccionados
  MIXTO : 1,036 ejemplos seleccionados

Total dataset balanceado: 8,236


In [4]:
# Split train / val / test estratificado por clase
from sklearn.model_selection import train_test_split

# Primero separar test
df_train_val, df_test = train_test_split(
    df_bal,
    test_size=TEST_RATIO,
    stratify=df_bal['label'],
    random_state=SEED,
)

# Luego separar val del train
val_ratio_adjusted = VAL_RATIO / (1 - TEST_RATIO)
df_train, df_val = train_test_split(
    df_train_val,
    test_size=val_ratio_adjusted,
    stratify=df_train_val['label'],
    random_state=SEED,
)

print('Splits generados:')
for nombre, split in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f'\n  {nombre} ({len(split):,} ejemplos):')
    for label, n in split['label'].value_counts().sort_index().items():
        print(f'    {label:6s}: {n:4,}')


Splits generados:

  train (6,588 ejemplos):
    MIXTO :  828
    NEG   : 1,920
    NEU   : 1,920
    POS   : 1,920

  val (824 ejemplos):
    MIXTO :  104
    NEG   :  240
    NEU   :  240
    POS   :  240

  test (824 ejemplos):
    MIXTO :  104
    NEG   :  240
    NEU   :  240
    POS   :  240


In [5]:
# Guardar CSVs
df_train.to_csv(OUTPUT_DIR / 'train.csv', index=False, encoding='utf-8')
df_val.to_csv(OUTPUT_DIR / 'val.csv',   index=False, encoding='utf-8')
df_test.to_csv(OUTPUT_DIR / 'test.csv', index=False, encoding='utf-8')

print('Archivos guardados:')
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.0f} KB)')


Archivos guardados:
  test.csv  (139 KB)
  train.csv  (1139 KB)
  val.csv  (140 KB)


In [6]:
# Calcular class weights para el entrenamiento (raíz cuadrada de frecuencia inversa — igual que FinBERT)
import torch

conteos = df_train['label_id'].value_counts().sort_index()
total   = len(df_train)
weights = []
print('Class weights (sqrt de frecuencia inversa):')
for label_id in sorted(LABEL_MAP.values()):
    label_name = [k for k, v in LABEL_MAP.items() if v == label_id][0]
    n     = conteos.get(label_id, 1)
    w     = (total / n) ** 0.5
    weights.append(w)
    print(f'  {label_name:6s} (id={label_id}): n={n:,}  weight={w:.3f}')

class_weights = torch.tensor(weights, dtype=torch.float)
torch.save(class_weights, OUTPUT_DIR / 'class_weights.pt')
print(f'\nGuardado: dataset/class_weights.pt')
print('\n✓ Dataset listo.')
print('  Siguiente paso: 03_finetuning.ipynb')


Class weights (sqrt de frecuencia inversa):
  POS    (id=0): n=1,920  weight=1.852
  NEG    (id=1): n=1,920  weight=1.852
  NEU    (id=2): n=1,920  weight=1.852
  MIXTO  (id=3): n=828  weight=2.821

Guardado: dataset/class_weights.pt

✓ Dataset listo.
  Siguiente paso: 03_finetuning.ipynb
